In [1]:
print("fa")

fa


In [2]:

pip install rasterio pandas numpy


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import rasterio
import pandas as pd
import numpy as np


# ======================== CONFIGURATION ======================== #
# All common bands between Sentinel-2 and Landsat-8
# Landsat 8 native resolution is 30m — we resample to 10m
REFLECTANCE_BANDS = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7']
QA_BANDS = ['QA_PIXEL']
TARGET_RESOLUTION = 10  # metres


def find_band_files(root_folder, sr_bands=REFLECTANCE_BANDS, qa_bands=QA_BANDS):
    """
    Search for Landsat 8 Level-2 .TIF band files under root_folder.

    Surface Reflectance files: *_SR_B1.TIF … *_SR_B7.TIF
    QA files:                  *_QA_PIXEL.TIF

    Returns:
        sr_files : dict  {band_name: file_path}   e.g. {'B1': '…SR_B1.TIF'}
        qa_files : dict  {qa_name: file_path}      e.g. {'QA_PIXEL': '…QA_PIXEL.TIF'}
    """
    sr_files = {}
    qa_files = {}
    for dirpath, _, filenames in os.walk(root_folder):
        for f in filenames:
            if not f.upper().endswith('.TIF'):
                continue
            f_upper = f.upper()
            # Surface-reflectance bands
            for band in sr_bands:
                if f"_SR_{band}.TIF" in f_upper:
                    sr_files[band] = os.path.join(dirpath, f)
            # QA bands
            for qa in qa_bands:
                if f"_{qa}.TIF" in f_upper:
                    qa_files[qa] = os.path.join(dirpath, f)
    return sr_files, qa_files


def read_band_raw(band_path):
    """
    Read a Landsat GeoTIFF band. Returns raw array, transform, shape.
    """
    with rasterio.open(band_path) as ds:
        band = ds.read(1).astype('float32')
        transform = ds.transform
        shape = band.shape
    return band, transform, shape


def normalize_reflectance(band_array):
    """
    Apply Landsat 8 Collection 2 Level-2 scale:
        Reflectance = pixel_value * 0.0000275 − 0.2
    Fill value (0) → NaN, then clip to [0, 1].
    """
    band_array[band_array == 0] = np.nan
    reflectance = band_array * 0.0000275 - 0.2
    reflectance = np.clip(reflectance, 0, 1)
    return reflectance


def resample_30m_to_10m_coords(transform_30m, shape_30m):
    """
    Build 10m coordinate arrays from a 30m grid.
    Each 30m pixel → 3x3 block of 10m pixels.
    Offsets: -10m, 0m, +10m from the 30m pixel center.

    Same approach as Sentinel-2 20m→10m resampling (±5m offsets with 2x2 repeat),
    but here 30m→10m uses ±10m offsets with 3x3 repeat.
    """
    rows_30, cols_30 = shape_30m
    t = transform_30m

    # 30m pixel centers
    xs_1d_30 = t.c + (np.arange(cols_30) + 0.5) * t.a   # easting
    ys_1d_30 = t.f + (np.arange(rows_30) + 0.5) * t.e   # northing

    # 10m x-coordinates: 3 sub-pixels per 30m pixel (-10, 0, +10)
    cols_10 = cols_30 * 3
    xs_10m = np.empty(cols_10, dtype=np.float32)
    xs_10m[0::3] = xs_1d_30 - 10.0
    xs_10m[1::3] = xs_1d_30
    xs_10m[2::3] = xs_1d_30 + 10.0

    # 10m y-coordinates: 3 sub-pixels per 30m pixel (+10, 0, -10)
    rows_10 = rows_30 * 3
    ys_10m = np.empty(rows_10, dtype=np.float32)
    ys_10m[0::3] = ys_1d_30 + 10.0
    ys_10m[1::3] = ys_1d_30
    ys_10m[2::3] = ys_1d_30 - 10.0

    return xs_10m, ys_10m, cols_10, rows_10


def resample_30m_to_10m_values(band_30m):
    """
    Resample 30m array to 10m using nearest-neighbour (3x3 repeat).
    Each 30m pixel value is copied into a 3x3 block of 10m pixels.
    """
    return np.repeat(np.repeat(band_30m, 3, axis=1), 3, axis=0)


def bands_to_dataframe(sr_files, qa_files):
    """
    Combine all Landsat bands into a single DataFrame with Longitude, Latitude,
    reflectance bands (B1–B7), and QA_PIXEL — all resampled from 30m to 10m.
    Uses chunk-based processing (same pattern as Sentinel-2 20m→10m resampling).
    
    NOTE: To avoid memory issues on Kaggle, use bands_to_csv() instead for large scenes.
    """
    if not sr_files:
        raise FileNotFoundError(
            "No SR band files found! Check root_folder and ensure *_SR_B*.TIF files exist."
        )

    # --- Read first band to get transform/shape ---
    first_path = next(iter(sr_files.values()))
    _, transform_30m, shape_30m = read_band_raw(first_path)
    rows_30, cols_30 = shape_30m

    # --- Build 10m coordinate grid ---
    xs_10m, ys_10m, cols_10, rows_10 = resample_30m_to_10m_coords(transform_30m, shape_30m)

    # --- Build DataFrame in chunks ---
    CHUNK = 50  # small chunks to save memory
    df_list = []

    for start in range(0, rows_30, CHUNK):
        end = min(start + CHUNK, rows_30)
        n = end - start

        # Three y-values per 30m row
        ys_chunk = np.empty(n * 3, dtype=np.float32)
        ys_chunk[0::3] = ys_10m[start * 3: start * 3 + n * 3: 3]
        ys_chunk[1::3] = ys_10m[start * 3 + 1: start * 3 + n * 3: 3]
        ys_chunk[2::3] = ys_10m[start * 3 + 2: start * 3 + n * 3: 3]

        n10 = n * 3
        lon = np.tile(xs_10m, n10)
        lat = np.repeat(ys_chunk, cols_10)

        chunk_data = {"Longitude": lon, "Latitude": lat}

        # SR bands — read slice, normalize, resample, store as float16
        for band_name, path in sr_files.items():
            with rasterio.open(path) as ds:
                blk = ds.read(1, window=rasterio.windows.Window(0, start, cols_30, n)).astype('float32')
            blk[blk == 0] = np.nan
            blk = blk * 0.0000275 - 0.2
            np.clip(blk, 0, 1, out=blk)
            vals = np.repeat(np.repeat(blk, 3, axis=1), 3, axis=0)
            chunk_data[band_name] = vals.ravel().astype('float16')
            del blk, vals

        # QA bands — read slice, resample
        for qa_name, path in qa_files.items():
            with rasterio.open(path) as ds:
                blk = ds.read(1, window=rasterio.windows.Window(0, start, cols_30, n))
            vals = np.repeat(np.repeat(blk, 3, axis=1), 3, axis=0)
            chunk_data[qa_name] = vals.ravel().astype('uint16')
            del blk, vals

        chunk_df = pd.DataFrame(chunk_data)
        del chunk_data

        # Drop nodata rows within chunk
        sr_cols = list(sr_files.keys())
        chunk_df.dropna(subset=sr_cols, how='all', inplace=True)
        df_list.append(chunk_df)
        del chunk_df

    df = pd.concat(df_list, ignore_index=True)
    del df_list
    return df


def bands_to_csv(sr_files, qa_files, output_csv, center_crop=0.2):
    """
    Stream Landsat bands (resampled 30m→10m) directly to a CSV file.
    Only processes the center crop of the image to reduce file size.
    
    Parameters:
        center_crop: Fraction of image to keep from the center (0.2 = 20%).
                     Crops both rows and columns, so 0.2 → ~4% of total pixels.
                     Use 1.0 to process the full image.
    """
    import gc

    if not sr_files:
        raise FileNotFoundError(
            "No SR band files found! Check root_folder and ensure *_SR_B*.TIF files exist."
        )

    # --- Get transform/shape from first band ---
    first_path = next(iter(sr_files.values()))
    _, transform_30m, shape_30m = read_band_raw(first_path)
    rows_30, cols_30 = shape_30m

    # --- Compute center crop window (in 30m pixels) ---
    crop_rows = int(rows_30 * center_crop)
    crop_cols = int(cols_30 * center_crop)
    row_start = (rows_30 - crop_rows) // 2
    row_end = row_start + crop_rows
    col_start = (cols_30 - crop_cols) // 2
    col_end = col_start + crop_cols

    print(f"Original image: {rows_30} x {cols_30} pixels (30m)")
    print(f"Center crop {int(center_crop*100)}%: rows [{row_start}:{row_end}], cols [{col_start}:{col_end}]")
    print(f"Crop size: {crop_rows} x {crop_cols} = {crop_rows*crop_cols:,} pixels (30m)")
    print(f"After 10m resampling: {crop_rows*3} x {crop_cols*3} = {crop_rows*crop_cols*9:,} pixels")

    # --- Build 10m coordinate grid for the cropped region only ---
    t = transform_30m
    xs_1d_30 = t.c + (np.arange(col_start, col_end) + 0.5) * t.a
    ys_1d_30 = t.f + (np.arange(row_start, row_end) + 0.5) * t.e

    cols_10_crop = crop_cols * 3
    xs_10m = np.empty(cols_10_crop, dtype=np.float32)
    xs_10m[0::3] = xs_1d_30 - 10.0
    xs_10m[1::3] = xs_1d_30
    xs_10m[2::3] = xs_1d_30 + 10.0

    rows_10_crop = crop_rows * 3
    ys_10m = np.empty(rows_10_crop, dtype=np.float32)
    ys_10m[0::3] = ys_1d_30 + 10.0
    ys_10m[1::3] = ys_1d_30
    ys_10m[2::3] = ys_1d_30 - 10.0

    sr_cols = list(sr_files.keys())
    qa_cols_list = list(qa_files.keys())

    CHUNK = 30  # 30m rows per chunk
    total_rows_written = 0
    first_write = True

    print(f"Streaming to {output_csv} in chunks of {CHUNK} rows ...")

    for start in range(0, crop_rows, CHUNK):
        end = min(start + CHUNK, crop_rows)
        n = end - start  # number of 30m rows in this chunk

        # Actual row position in the original image
        abs_row = row_start + start

        # 10m y-coordinates for this chunk
        ys_chunk = np.empty(n * 3, dtype=np.float32)
        ys_chunk[0::3] = ys_10m[start * 3: start * 3 + n * 3: 3]
        ys_chunk[1::3] = ys_10m[start * 3 + 1: start * 3 + n * 3: 3]
        ys_chunk[2::3] = ys_10m[start * 3 + 2: start * 3 + n * 3: 3]

        n10 = n * 3
        lon = np.tile(xs_10m, n10)
        lat = np.repeat(ys_chunk, cols_10_crop)

        chunk_data = {"Longitude": lon, "Latitude": lat}

        # SR bands — read only the crop window from disk
        for band_name, path in sr_files.items():
            with rasterio.open(path) as ds:
                blk = ds.read(1, window=rasterio.windows.Window(
                    col_start, abs_row, crop_cols, n
                )).astype('float32')
            blk[blk == 0] = np.nan
            blk = blk * 0.0000275 - 0.2
            np.clip(blk, 0, 1, out=blk)
            vals = np.repeat(np.repeat(blk, 3, axis=1), 3, axis=0)
            chunk_data[band_name] = vals.ravel()
            del blk, vals

        # QA bands
        for qa_name, path in qa_files.items():
            with rasterio.open(path) as ds:
                blk = ds.read(1, window=rasterio.windows.Window(
                    col_start, abs_row, crop_cols, n
                ))
            vals = np.repeat(np.repeat(blk, 3, axis=1), 3, axis=0)
            chunk_data[qa_name] = vals.ravel().astype('uint16')
            del blk, vals

        chunk_df = pd.DataFrame(chunk_data)
        del chunk_data

        # Drop nodata rows
        chunk_df.dropna(subset=sr_cols, how='all', inplace=True)

        # Append to CSV
        chunk_df.to_csv(output_csv, mode='a' if not first_write else 'w',
                        header=first_write, index=False)
        total_rows_written += len(chunk_df)
        first_write = False
        del chunk_df
        gc.collect()

        # Progress
        pct = min(100, int(end / crop_rows * 100))
        print(f"  Progress: {pct}%  ({total_rows_written:,} rows written)", end='\r')

    print(f"\nDone! Saved {total_rows_written:,} rows to {output_csv}")


# ======================== USAGE ======================== #
if __name__ == "__main__":

    # ---------- Set your root folder ----------
    # On Kaggle:
    root_folder = "/kaggle/input/datasets/mezbaussalaheen/usgs-landsat/LC08_L2SP_137043_20131106_20200912_02_T1"
    # On local Windows:
    # root_folder = r"E:\3-2\Capstone\LC08_L2SP_137043_20131106_20200912_02_T1"

    # Landsat 8 OLI — Common bands with Sentinel-2:
    # B1  = Coastal/Aerosol  (↔ Sentinel-2 B01)
    # B2  = Blue             (↔ Sentinel-2 B02)
    # B3  = Green            (↔ Sentinel-2 B03)
    # B4  = Red              (↔ Sentinel-2 B04)
    # B5  = NIR              (↔ Sentinel-2 B08)   ⭐
    # B6  = SWIR-1           (↔ Sentinel-2 B11)
    # B7  = SWIR-2           (↔ Sentinel-2 B12)   ⭐
    # QA_PIXEL = Cloud/quality mask (↔ Sentinel-2 SCL)

    # Step 1: Find band files
    sr_files, qa_files = find_band_files(root_folder)
    print("Surface Reflectance bands found:")
    for b, p in sr_files.items():
        print(f"  {b}: {p}")
    print("QA bands found:")
    for b, p in qa_files.items():
        print(f"  {b}: {p}")

    # Step 2: Stream to CSV (center 20% crop, resampled to 10m)
    output_csv = "/kaggle/working/landsat8_10m_all_bands.csv"
    print(f"\nResampling center 20% from 30m → {TARGET_RESOLUTION}m ...")
    bands_to_csv(sr_files, qa_files, output_csv, center_crop=0.2)

    # Step 3: Load and preview (only first few rows to save memory)
    df_preview = pd.read_csv(output_csv, nrows=5)
    print(f"\nColumns: {list(df_preview.columns)}")
    print(df_preview)
    # Columns: Longitude, Latitude, B1, B2, B3, B4, B5, B6, B7, QA_PIXEL

    # To load the full CSV later for ML:
    # df = pd.read_csv(output_csv)


Surface Reflectance bands found:
  B3: /kaggle/input/datasets/mezbaussalaheen/usgs-landsat/LC08_L2SP_137043_20131106_20200912_02_T1/LC08_L2SP_137043_20131106_20200912_02_T1_SR_B3.TIF
  B2: /kaggle/input/datasets/mezbaussalaheen/usgs-landsat/LC08_L2SP_137043_20131106_20200912_02_T1/LC08_L2SP_137043_20131106_20200912_02_T1_SR_B2.TIF
  B1: /kaggle/input/datasets/mezbaussalaheen/usgs-landsat/LC08_L2SP_137043_20131106_20200912_02_T1/LC08_L2SP_137043_20131106_20200912_02_T1_SR_B1.TIF
  B5: /kaggle/input/datasets/mezbaussalaheen/usgs-landsat/LC08_L2SP_137043_20131106_20200912_02_T1/LC08_L2SP_137043_20131106_20200912_02_T1_SR_B5.TIF
  B4: /kaggle/input/datasets/mezbaussalaheen/usgs-landsat/LC08_L2SP_137043_20131106_20200912_02_T1/LC08_L2SP_137043_20131106_20200912_02_T1_SR_B4.TIF
  B7: /kaggle/input/datasets/mezbaussalaheen/usgs-landsat/LC08_L2SP_137043_20131106_20200912_02_T1/LC08_L2SP_137043_20131106_20200912_02_T1_SR_B7.TIF
  B6: /kaggle/input/datasets/mezbaussalaheen/usgs-landsat/LC08_L2SP